# NTPC PPE Detection PoC: Stage 2 PPE Detector Training

This notebook trains a YOLOv11n object detector on the cropped person dataset prepared in the previous notebook. The model will run locally on your RTX 4070 GPU.

In [1]:
# 1. Import dependencies and check GPU status
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / (1024**3), "GB")

PyTorch version: 2.5.1+cu121
CUDA Available: True
GPU Device Name: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM: 7.99560546875 GB


In [2]:
# 2. Initialize YOLOv11 model for fine-tuning
# yolo11n (nano) is fastest — ideal for real-time FPS target
# Ultralytics auto-downloads the pretrained weights if not already cached
model = YOLO('yolo11n.pt')

In [ ]:
# 3. Configure and start fine-tuning
# Optimized for RTX 4070 (8 GB VRAM) with Tensor Core FP16 via AMP (on by default)
results = model.train(
    data='../datasets/ppe_crops/data.yaml',
    epochs=30,             # Sufficient for fine-tuning on pre-trained weights
    imgsz=320,             # Reduced from 640 — fast inference on small person crops
    batch=128,             # Fits RTX 4070 VRAM at imgsz=320 (optimized for speed)
    device=0,              # GPU index 0
    workers=0,             # Windows-safe (spawn multiprocessing hangs with >0)
    cache=True,            # Cache images in RAM — removes disk I/O bottleneck
    val=True,              # Enable per-epoch validation for early stopping
    patience=10,           # Stop early if val mAP stalls for 10 epochs
    lr0=0.01,              # Initial learning rate
    optimizer='AdamW',     # Modern optimizer with weight decay
    cos_lr=True,           # Cosine LR schedule — smoother convergence
    save=True,             # Save checkpoints
    project='ntpc_ppe',    # Project folder
    name='stage2_crop_det' # Training run name
)

# Print save directory for the resume / export cells
print(f'\nTraining complete. Weights saved to: {results.save_dir}')

Ultralytics 8.4.75  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=../datasets/ppe_crops/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=stage2_crop_det-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW,

KeyboardInterrupt: 

In [ ]:
# 3b. Resume training from the latest checkpoint (if interrupted)
import glob

last_pts = sorted(glob.glob('ntpc_ppe/stage2_crop_det*/weights/last.pt'))
if not last_pts:
    print('No checkpoint found to resume from. Run Cell 3 first.')
else:
    resume_path = last_pts[-1]
    print(f'Resuming from: {resume_path}')
    model = YOLO(resume_path)
    results = model.train(resume=True, workers=0)
    print(f'Resumed training complete. Weights at: {results.save_dir}')

In [ ]:
# 4. Validate Model & Print Metrics
metrics = model.val()
print(f'Validation mAP@50:     {metrics.box.map50:.4f}')
print(f'Validation mAP@50-95:  {metrics.box.map:.4f}')
print(f'Precision (per class): {[f"{p:.3f}" for p in metrics.box.p]}')
print(f'Recall    (per class): {[f"{r:.3f}" for r in metrics.box.r]}')

In [ ]:
# 5. Save and Export model weights to models/ for the backend pipeline
import os
import shutil
import glob

models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Find best.pt from the latest training run via glob (kernel-restart safe)
best_candidates = sorted(glob.glob('ntpc_ppe/stage2_crop_det*/weights/best.pt'))

if best_candidates:
    best_weights = best_candidates[-1]
    target_weights = os.path.join(models_dir, 'ppe_crop_detector.pt')
    shutil.copy(best_weights, target_weights)
    print(f'Copied: {best_weights} -> {target_weights}')
else:
    # Fallback: try using model.trainer if still in memory
    try:
        best_weights = os.path.join(model.trainer.save_dir, 'weights', 'best.pt')
        if os.path.exists(best_weights):
            target_weights = os.path.join(models_dir, 'ppe_crop_detector.pt')
            shutil.copy(best_weights, target_weights)
            print(f'Copied (from trainer): {best_weights} -> {target_weights}')
        else:
            print(f'Error: best.pt not found at {best_weights}')
    except AttributeError:
        print('Error: No training run found. Run Cell 3 first, then re-run this cell.')